### ***Import Libraries & Load Dataset***

In [32]:
import pandas as pd
import numpy as np

# Load your dataset
df = pd.read_excel("ML_Preprocessing_Practice_Dataset.excel")  

In [33]:
df.head()

,Education_Level,City,Department,Employment_Type,Age,Monthly_Income,Performance
0,High School,Hyderabad,Marketing,Part-Time,27.0,21250.0,Average
1,Bachelor,Chennai,Sales,Part-Time,24.0,45231.0,Average
2,High School,Chennai,Sales,Contract,36.0,25366.0,Average
3,High School,Bengaluru,IT,Contract,45.0,22289.0,Average
4,High School,Bengaluru,IT,Full-Time,26.0,24658.0,Average


In [34]:
print(df.isnull().sum())

Education_Level     6
City               13
Department         11
Employment_Type     8
Age                15
Monthly_Income      7
Performance         0
dtype: int64


In [35]:
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Monthly_Income'] = df['Monthly_Income'].fillna(df['Monthly_Income'].mean())

df['City'] = df['City'].fillna(df['City'].mode()[0])
df['Department'] = df['Department'].fillna(df['Department'].mode()[0])
df['Employment_Type'] = df['Employment_Type'].fillna(df['Employment_Type'].mode()[0])
df['Education_Level'] = df['Education_Level'].fillna(df['Education_Level'].mode()[0])


print(df.isnull().sum())

Education_Level    0
City               0
Department         0
Employment_Type    0
Age                0
Monthly_Income     0
Performance        0
dtype: int64


#### ***dentify the Nominal Categorical Columns***

- Before applying One-Hot Encoding, we first identify the columns that contain nominal categorical data.
- In our dataset:
    - City
    - Department
    - Employment_Type
- These columns have no natural order, so they are suitable for One-Hot Encoding.

In [36]:
nominal_cols = ['City', 'Department', 'Employment_Type']

# ***Step 1: One-Hot Encoding — PANDAS Method (get_dummies)***

In [13]:
df_pandas_ohe = pd.get_dummies(df, columns=nominal_cols, dtype=int)

# df → The original DataFrame.
# columns=nominal_cols → Specifies the columns on which One-Hot Encoding should be applied.
# dtype=int → Creates binary columns with integer values (0 and 1) instead of boolean values (True and False).

In [14]:
df_pandas_ohe.head()

,Education_Level,Age,Monthly_Income,Performance,City_Bengaluru,City_Chennai,City_Delhi,City_Hyderabad,City_Mumbai,Department_HR,Department_IT,Department_Marketing,Department_Sales,Employment_Type_Contract,Employment_Type_Full-Time,Employment_Type_Part-Time
0,High School,27.0,21250.0,Average,False,False,False,True,False,False,False,True,False,False,False,True
1,Bachelor,24.0,45231.0,Average,False,True,False,False,False,False,False,False,True,False,False,True
2,High School,36.0,25366.0,Average,False,True,False,False,False,False,False,False,True,True,False,False
3,High School,45.0,22289.0,Average,True,False,False,False,False,False,True,False,False,True,False,False
4,High School,26.0,24658.0,Average,True,False,False,False,False,False,True,False,False,False,True,False


### ***What Happened?***

- The original columns:
    - City
    - Department
    - Employment_Type
- have been removed and replaced with multiple binary columns.
- For example

- Instead of: `City`
    - Hyderabad
    - Delhi
    - Mumbai
-  we now have:
  - City_Hyderabad
  - City_Delhi
  - City_Mumbai

- Similarly : The `Department` column is converted into:
    - Department_IT
    - Department_HR
    - Department_Sales
    - Department_Marketing
- The `Employment_Type` column is converted into:
    - Employment_Type_Full-Time
    - Employment_Type_Part-Time
    - Employment_Type_Contract
- Each category gets its own binary column.

# ***K-1 Encoding***

In [16]:
df_pandas_ohe =  pd.get_dummies(
    df,
    columns=nominal_cols,
    drop_first=True,
    dtype=int
)

In [17]:
df_pandas_ohe.head()

,Education_Level,Age,Monthly_Income,Performance,City_Chennai,City_Delhi,City_Hyderabad,City_Mumbai,Department_IT,Department_Marketing,Department_Sales,Employment_Type_Full-Time,Employment_Type_Part-Time
0,High School,27.0,21250.0,Average,0,0,1,0,0,1,0,0,1
1,Bachelor,24.0,45231.0,Average,1,0,0,0,0,0,1,0,1
2,High School,36.0,25366.0,Average,1,0,0,0,0,0,1,0,0
3,High School,45.0,22289.0,Average,0,0,0,0,1,0,0,0,0
4,High School,26.0,24658.0,Average,0,0,0,0,1,0,0,1,0


In [19]:
df['Department'].unique()

array(['Marketing', 'Sales', 'IT', 'HR'], dtype=object)

# ***Step 2 : One-Hot Encoding Using `OneHotEncoder` (Scikit-learn)***



- In real Machine Learning projects, **One-Hot Encoding should be applied after performing the train-test split**.
- This approach prevents **data leakage** because the encoder learns the categories only from the training data.
- However, in this section, our objective is **only to understand how the `OneHotEncoder` class works**. To keep the implementation simple and focus on - the encoding process, we will apply it directly to the entire dataset.
- In the complete Machine Learning pipeline, we will follow the correct approach by applying the encoder **after** the train-test split.


In [37]:
from sklearn.preprocessing import OneHotEncoder

In [38]:
encoder = OneHotEncoder( drop="first",
                        sparse_output=False, 
                        dtype=int, 
                        handle_unknown="ignore" 
                       )

# sparse_output=False returns the encoded data as a NumPy array instead of a sparse matrix.
# If this parameter is not specified, the output is returned as a sparse matrix to save memory.

In [39]:
encoded_data = encoder.fit_transform(df[nominal_cols])

In [40]:
encoded_data

array([[0, 0, 1, ..., 0, 0, 1],
       [1, 0, 0, ..., 1, 0, 1],
       [1, 0, 0, ..., 1, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 1],
       [0, 0, 0, ..., 0, 0, 0]])

In [41]:
encoder.get_feature_names_out(nominal_cols)

array(['City_Chennai', 'City_Delhi', 'City_Hyderabad', 'City_Mumbai',
       'Department_IT', 'Department_Marketing', 'Department_Sales',
       'Employment_Type_Full-Time', 'Employment_Type_Part-Time'],
      dtype=object)

In [42]:
encoded_df = pd.DataFrame( encoded_data, 
                          columns=encoder.get_feature_names_out(nominal_cols), 
                          index=df.index 
                         )

In [43]:
encoded_df.head()

,City_Chennai,City_Delhi,City_Hyderabad,City_Mumbai,Department_IT,Department_Marketing,Department_Sales,Employment_Type_Full-Time,Employment_Type_Part-Time
0,0,0,1,0,0,1,0,0,1
1,1,0,0,0,0,0,1,0,1
2,1,0,0,0,0,0,1,0,0
3,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,1,0


In [44]:
df = df.drop(columns=nominal_cols)

In [45]:
df = pd.concat([df, encoded_df], axis=1)

In [46]:
df.head()

,Education_Level,Age,Monthly_Income,Performance,City_Chennai,City_Delhi,City_Hyderabad,City_Mumbai,Department_IT,Department_Marketing,Department_Sales,Employment_Type_Full-Time,Employment_Type_Part-Time
0,High School,27.0,21250.0,Average,0,0,1,0,0,1,0,0,1
1,Bachelor,24.0,45231.0,Average,1,0,0,0,0,0,1,0,1
2,High School,36.0,25366.0,Average,1,0,0,0,0,0,1,0,0
3,High School,45.0,22289.0,Average,0,0,0,0,1,0,0,0,0
4,High School,26.0,24658.0,Average,0,0,0,0,1,0,0,1,0
